# 006 · Type Conversion and Checking

> **Syntax › Data Types** · Python 3.14

Python is **dynamically but strongly typed**: variables don't have types, values do — and Python will not silently convert between unrelated types (no `"a" + 1`). Conversions are explicit calls to the type itself, and type checks happen at runtime with `type()` and `isinstance()`.

**In this notebook you will learn:**

- converting with `int()`, `float()`, `str()`, `bool()` vs C#'s casts, `Convert`, `Parse`, and `TryParse`
- handling failed conversions (`ValueError`) and writing a `try_parse` helper
- which conversions happen implicitly (numeric promotion) and which never do
- `type()` vs `isinstance()` vs C#'s `GetType()` and `is`
- duck typing, and the EAFP vs LBYL coding styles

## C# mental model

| C# | Python |
|---|---|
| `int.Parse("42")` / `Convert.ToInt32("42")` | `int("42")` |
| `double.Parse("3.5")` | `float("3.5")` |
| `(int)3.9` | `int(3.9)` → `3` |
| `x.ToString()` | `str(x)` |
| `int.TryParse(s, out var n)` | `try: int(s) except ValueError:` |
| `FormatException` | `ValueError` |
| `x.GetType() == typeof(int)` | `type(x) is int` |
| `x is int` / `x is int n` | `isinstance(x, int)` |
| `x as string` | no equivalent — check with `isinstance` |

The key difference: C# checks types at **compile time**; Python checks them at **runtime**, when an operation actually happens.

## Converting with constructors

In Python the type *is* the conversion function. `int(x)`, `float(x)`, `str(x)` and `bool(x)` construct a new value from `x`.

```csharp
// C#
int n = int.Parse("42");
double d = double.Parse("3.5", CultureInfo.InvariantCulture);
string s = 42.ToString();
int t = (int)3.9;         // 3
```

In [ ]:
print(int("42"), int("  -7  "), int("1_000"))   # whitespace and underscores are OK
print(float("3.5"), float("1e-3"), float("inf"))
print(str(42), str(3.14), str([1, 2]))
print(int(3.9), int(-3.9))                     # truncation toward zero, like (int) cast
print(int(True), float(False))                 # 1 0.0

Unlike `double.Parse`, `float()` is **culture-independent**: it always uses `.` as the decimal separator and never accepts thousands separators.

## When conversion fails: `ValueError`

A bad string raises `ValueError` — the counterpart of C#'s `FormatException`. A wrong *type* of argument raises `TypeError`.

In [ ]:
for bad in ["3.5", "abc", "", "1,000"]:
    try:
        int(bad)
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

try:
    int(None)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `int("3.5")` fails even though `int(3.5)` works — `int()` on a string expects an integer literal. Go via `float` first if you need it: `int(float("3.5"))`.

## No `TryParse` — write a helper

There is no `TryParse` and no `out` parameters. The Pythonic approach is to try the conversion and catch the exception. If you do it often, wrap it in a helper that returns `None` on failure:

```csharp
// C#
if (int.TryParse(input, out var qty))
    Console.WriteLine(qty * 2);
else
    Console.WriteLine("not a number");
```

In [ ]:
def try_parse_int(text: str) -> int | None:
    try:
        return int(text)
    except ValueError:
        return None

for text in ["12", "x12", "007"]:
    if (qty := try_parse_int(text)) is not None:     # walrus := assigns and tests
        print(f"{text!r} -> {qty * 2}")
    else:
        print(f"{text!r} -> not a number")

A generic version works for any converter — `int`, `float`, `Decimal`, `date.fromisoformat`, ... — using a PEP 695 type parameter:

In [ ]:
from collections.abc import Callable
from datetime import date

def try_parse[T](text: str, convert: Callable[[str], T], default: T | None = None) -> T | None:
    try:
        return convert(text)
    except (ValueError, TypeError):    # Python 3.14 also allows: except ValueError, TypeError:
        return default

print(try_parse("2.5", float))
print(try_parse("oops", float, 0.0))
print(try_parse("2025-01-31", date.fromisoformat))
print(try_parse("2025-02-31", date.fromisoformat))   # invalid date -> None

## `bool()` and `str()` conversions

`bool(x)` uses truthiness *(see Data Types › 004 · Booleans and Truthiness)*, so it does **not** parse text. `str(x)` works on any object, calling its `__str__` — like `ToString()`.

In [ ]:
print(bool("False"), bool(""), bool(0), bool([0]))   # True False False True
print(str(None), str(True), str(3 + 4j))

Other useful converters:

In [ ]:
from decimal import Decimal

print(int("ff", 16), int("0b101", 0))     # parse other bases
print(Decimal("19.99"))                    # string -> Decimal (exact)
print(list("abc"), tuple([1, 2]), set([1, 1, 2]))   # collection conversions
print(ord("A"), chr(66))                   # char code <-> 1-char str

## Implicit conversions: numeric promotion only

Python only converts implicitly **between numeric types**, always widening: `bool` → `int` → `float` → `complex`. This mirrors C#'s implicit numeric conversions (`int` → `double`).

In [ ]:
print(1 + 2.0, type(1 + 2.0))          # 3.0 float
print(True + 1)                        # 2   — bool is an int
print(2 + 3j)                          # (2+3j) — int -> complex
print(10 / 4, type(10 / 4))            # 2.5 float — / always produces float

Everything else must be converted explicitly. In particular — unlike C#, where `"Total: " + 5` calls `ToString()` for you — Python refuses to mix `str` and numbers:

In [ ]:
count = 5
try:
    print("Total: " + count)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("Total: " + str(count))     # explicit
print(f"Total: {count}")          # idiomatic

⚠️ **Gotcha:** comparing a string to a number is not an error with `==` — it is simply `False`. Ordering comparisons (`<`, `>`) raise `TypeError`.

In [ ]:
print("5" == 5)    # False — no conversion happens

try:
    "5" < 10
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Checking types: `type()` vs `isinstance()`

- `type(x)` returns the exact class — like `x.GetType()`.
- `isinstance(x, T)` also accepts **subclasses** — like C#'s `x is T`. This is almost always what you want.

```csharp
// C#
object x = 42;
Console.WriteLine(x.GetType() == typeof(int));   // True
Console.WriteLine(x is int);                     // True
```

In [ ]:
x = 42
print(type(x))                 # <class 'int'>
print(type(x) is int)          # True  — exact match (C#: GetType() == typeof(int))
print(isinstance(x, int))      # True  (C#: x is int)
print(type(x).__name__)        # int — C#: x.GetType().Name

The difference shows up with subclasses. `bool` is a subclass of `int`:

In [ ]:
flag = True
print(type(flag) is int)        # False — exact type is bool
print(isinstance(flag, int))    # True  — a bool *is an* int

class Animal: pass
class Dog(Animal): pass

d = Dog()
print(type(d) is Animal, isinstance(d, Animal))   # False True
print(issubclass(Dog, Animal))                    # True (C#: typeof(Animal).IsAssignableFrom(typeof(Dog)))

### Checking against several types

`isinstance` accepts a **tuple** of types, or a **union** written with `|` (Python 3.10+):

In [ ]:
def describe(value) -> str:
    if isinstance(value, bool):          # check bool first — it's also an int!
        return "boolean"
    if isinstance(value, int | float):   # union type
        return "number"
    if isinstance(value, (str, bytes)):  # tuple of types
        return "text"
    return f"other ({type(value).__name__})"

for v in [True, 3, 2.5, "hi", b"raw", [1], None]:
    print(f"{v!r:>6}: {describe(v)}")

For type-based branching, `match` with class patterns is often cleaner — the equivalent of a C# `switch` with type patterns *(see Control Flow › 002 · The match Statement)*:

In [ ]:
def describe_match(value) -> str:
    match value:
        case bool():
            return "boolean"
        case int() | float():
            return "number"
        case str(s) if not s:
            return "empty text"
        case str():
            return "text"
        case _:
            return "other"

print([describe_match(v) for v in [False, 7, "", "x", None]])

## Duck typing

Python code usually doesn't check types at all. If an object *behaves* like what you need — has the right methods — it works. "If it walks like a duck and quacks like a duck, it's a duck." This is like C# generics constrained to an interface, except no interface declaration is needed. *(The typed version of this idea is `Protocol` — see Classes & OOP › 006 · Abstract Classes and Protocols.)*

In [ ]:
class Duck:
    def speak(self) -> str:
        return "Quack"

class Robot:
    def speak(self) -> str:
        return "Beep"

def make_it_speak(thing) -> None:
    print(thing.speak())     # no isinstance check — just call it

make_it_speak(Duck())
make_it_speak(Robot())       # unrelated class, still works

Built-in functions are duck-typed too: `len()` works on anything with `__len__`, `for` works on anything iterable, `sum()` works on any iterable of numbers.

In [ ]:
def total_length(items) -> int:
    return sum(len(x) for x in items)

print(total_length(["ab", "cde"]))          # strings
print(total_length([[1, 2], (3,), {4: 5}])) # list, tuple, dict — all have a length

## EAFP vs LBYL

Two styles for handling things that might fail:

- **LBYL** — *Look Before You Leap*: check first, then act. Typical C# style (`if (dict.ContainsKey(k))`, `TryParse`).
- **EAFP** — *Easier to Ask Forgiveness than Permission*: just try it and handle the exception. Idiomatic Python.

EAFP avoids race conditions (the check and the action can't drift apart) and duplicate lookups, and in Python exceptions are cheap enough to use for control flow like this.

In [ ]:
stock = {"pen": 10, "cup": 0}

# LBYL — C#-style
if "lamp" in stock:
    print(stock["lamp"])
else:
    print("LBYL: not stocked")

# EAFP — Pythonic
try:
    print(stock["lamp"])
except KeyError:
    print("EAFP: not stocked")

In [ ]:
# EAFP with conversion: accept anything int() can handle
def to_quantity(value) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        raise ValueError(f"invalid quantity: {value!r}") from None

print(to_quantity("3"), to_quantity(4.0), to_quantity(True))
try:
    to_quantity("three")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** EAFP is not a licence to catch everything. Keep the `try` block small and catch the **specific** exception — a bare `except:` hides real bugs. *(More in Error Handling › 001 · try / except / finally.)*

## Summary

| Task | C# | Python |
|---|---|---|
| Parse int | `int.Parse(s)` | `int(s)` |
| Parse float | `double.Parse(s, Invariant)` | `float(s)` |
| Try parse | `int.TryParse(s, out var n)` | `try: int(s) except ValueError:` |
| Truncate | `(int)3.9` | `int(3.9)` |
| To string | `x.ToString()` | `str(x)` / f-string |
| Parse error | `FormatException` | `ValueError` |
| Exact type | `x.GetType() == typeof(T)` | `type(x) is T` |
| Is-a check | `x is T` | `isinstance(x, T)` |
| Several types | `x is int or double` | `isinstance(x, int \| float)` |
| Subclass check | `typeof(B).IsAssignableFrom(typeof(D))` | `issubclass(D, B)` |
| Type name | `x.GetType().Name` | `type(x).__name__` |
| Style | LBYL (`TryGetValue`, `TryParse`) | EAFP (`try` / `except`) |

## Exercises

1. Write `try_parse_float(text)` that returns a `float` or `None`, and also accepts a comma as decimal separator (`"3,5"` → `3.5`).
2. Given `raw = ["10", "x", "3.5", "", "7"]`, produce the list of values that parse as **ints**: `[10, 7]`.
3. Write `kind(x)` that returns `"int"` for ints (but **not** bools), `"bool"` for bools, `"str"` for strings, and `"unknown"` otherwise — using `isinstance`.
4. Rewrite this LBYL code in EAFP style:
   ```python
   if "qty" in order and order["qty"].isdigit():
       qty = int(order["qty"])
   else:
       qty = 0
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def try_parse_float(text: str) -> float | None:
    try:
        return float(text.replace(",", "."))
    except ValueError:
        return None

print(try_parse_float("3,5"), try_parse_float("abc"))   # 3.5 None

# 2
def try_parse_int(text: str) -> int | None:
    try:
        return int(text)
    except ValueError:
        return None

raw = ["10", "x", "3.5", "", "7"]
print([n for s in raw if (n := try_parse_int(s)) is not None])   # [10, 7]

# 3
def kind(x) -> str:
    if isinstance(x, bool):
        return "bool"
    if isinstance(x, int):
        return "int"
    if isinstance(x, str):
        return "str"
    return "unknown"

print(kind(1), kind(True), kind("a"), kind(1.0))   # int bool str unknown

# 4
order = {"qty": "5"}
try:
    qty = int(order["qty"])
except (KeyError, ValueError):
    qty = 0
print(qty)
```

</details>